In [1]:
# Assignment 5 - Reproducibility
# Paso 1: Crear el dataset breast_cancer.csv

from sklearn.datasets import load_breast_cancer
import hashlib

# Cargar el dataset
data = load_breast_cancer(as_frame=True)

# Obtener características y variable objetivo
df = data.frame

# Guardar los datos en CSV
df.to_csv("breast_cancer.csv", index=False)

# Calcular la huella digital SHA-256
with open("breast_cancer.csv", "rb") as archivo:
    fingerprint = hashlib.sha256(archivo.read()).hexdigest()

# Mostrar resultados
print("Dataset creado correctamente.")
print("Archivo: breast_cancer.csv")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("SHA-256:", fingerprint)

Dataset creado correctamente.
Archivo: breast_cancer.csv
Filas: 569
Columnas: 31
SHA-256: 75a1d74a59df9cb0a78ecc0a36085bf8a85239557f178d3b8d758e88b26416e1


In [3]:
%%writefile train.py
# Paso 2: Entrenamiento inicial del modelo

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Cargar el dataset
df = pd.read_csv("breast_cancer.csv")

# Separar características y variable objetivo
X = df.drop(columns=["target"])
y = df["target"]

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Crear un pipeline para evitar data leakage
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Entrenar el modelo
pipeline.fit(X_train, y_train)

# Evaluar el modelo
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Entrenamiento completado")
print(f"Accuracy: {accuracy:.4f}")

Writing train.py


In [4]:
# Paso 3: Reproducibilidad con 5 semillas
# Assignment 5 - Reproducibility

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Cargar los datos
df = pd.read_csv("breast_cancer.csv")

X = df.drop(columns=["target"])
y = df["target"]

# Cinco semillas para evaluar la variabilidad
seeds = [0, 12, 21, 42, 123]
resultados = []

for seed in seeds:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=seed
        ))
    ])

    pipeline.fit(X_train, y_train)

    predicciones = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predicciones)

    resultados.append({
        "seed": seed,
        "accuracy": accuracy
    })

# Mostrar resultados
resultados_df = pd.DataFrame(resultados)

media = resultados_df["accuracy"].mean()
desviacion = resultados_df["accuracy"].std()

print("RESULTADOS DE LAS 5 SEMILLAS")
print(resultados_df.to_string(index=False))

print(f"\nAccuracy promedio: {media:.4f}")
print(f"Desviación estándar: {desviacion:.4f}")

# Guardar resultados
resultados_df.to_csv("results.csv", index=False)

print("\nArchivo results.csv creado correctamente.")

RESULTADOS DE LAS 5 SEMILLAS
 seed  accuracy
    0  0.982456
   12  0.991228
   21  0.991228
   42  0.982456
  123  0.973684

Accuracy promedio: 0.9842
Desviación estándar: 0.0073

Archivo results.csv creado correctamente.


In [5]:
# Paso 4: Instalación de MLflow
# Assignment 5 - Reproducibility

!pip -q install mlflow

import mlflow

print("MLflow instalado correctamente.")
print("Versión de MLflow:", mlflow.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# Paso 5: Registro de experimentos con MLflow
# Assignment 5 - Reproducibility

import mlflow
import pandas as pd

# Configurar almacenamiento SQLite para MLflow
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")
mlflow.set_experiment("Breast_Cancer_Reproducibility")

# Cargar los resultados obtenidos anteriormente
resultados = pd.read_csv("results.csv")

# Registrar cada una de las cinco ejecuciones
for _, fila in resultados.iterrows():

    seed = int(fila["seed"])
    accuracy = float(fila["accuracy"])

    with mlflow.start_run(run_name=f"seed_{seed}"):

        mlflow.log_param("seed", seed)
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("scaler", "StandardScaler")
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("max_iter", 1000)

        mlflow.log_metric("accuracy", accuracy)

        print(f"Semilla {seed}: Accuracy = {accuracy:.4f}")

# Registrar el resumen estadístico
with mlflow.start_run(run_name="summary_5_seeds"):

    mlflow.log_metric(
        "accuracy_mean",
        resultados["accuracy"].mean()
    )

    mlflow.log_metric(
        "accuracy_std",
        resultados["accuracy"].std()
    )

    mlflow.log_param(
        "number_of_seeds",
        len(resultados)
    )

print("\nCinco ejecuciones registradas en MLflow.")
print("Resumen estadístico registrado correctamente.")

2026/10/09 22:54:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/09 22:54:14 INFO mlflow.store.db.utils: Updating database tables
2026/10/09 22:54:17 INFO mlflow.tracking.fluent: Experiment with name 'Breast_Cancer_Reproducibility' does not exist. Creating a new experiment.


Semilla 0: Accuracy = 0.9825
Semilla 12: Accuracy = 0.9912
Semilla 21: Accuracy = 0.9912
Semilla 42: Accuracy = 0.9825
Semilla 123: Accuracy = 0.9737

Cinco ejecuciones registradas en MLflow.
Resumen estadístico registrado correctamente.


In [9]:
!ls /content

breast_cancer.csv  mlflow.db  results.csv  sample_data	train.py


In [10]:
# Paso 7: Ejecutar train.py para entrenar con 5 semillas y registrar los resultados en MLflow

%%writefile train.py
# Assignment 5 - Reproducibility in Machine Learning
# Entrenamiento reproducible con cinco semillas y MLflow

from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# 1. Ubicar los archivos del proyecto
PROJECT_DIR = Path(__file__).resolve().parent
DATA_FILE = PROJECT_DIR / "breast_cancer.csv"
RESULTS_FILE = PROJECT_DIR / "results.csv"
MLFLOW_DB = PROJECT_DIR / "mlflow.db"

# 2. Configurar MLflow
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")
mlflow.set_experiment("Breast_Cancer_Reproducibility")

# 3. Cargar los datos
df = pd.read_csv(DATA_FILE)

X = df.drop(columns=["target"])
y = df["target"]

# 4. Definir las cinco semillas
seeds = [0, 12, 21, 42, 123]
resultados = []

# 5. Entrenar y evaluar con cada semilla
for seed in seeds:

    np.random.seed(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=seed
        ))
    ])

    pipeline.fit(X_train, y_train)

    predicciones = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predicciones)

    resultados.append({
        "seed": seed,
        "accuracy": accuracy
    })

    # Registrar la ejecución en MLflow
    with mlflow.start_run(run_name=f"train_seed_{seed}"):

        mlflow.log_param("seed", seed)
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("scaler", "StandardScaler")
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("max_iter", 1000)

        mlflow.log_metric("accuracy", float(accuracy))

# 6. Calcular las estadísticas
resultados_df = pd.DataFrame(resultados)

media = resultados_df["accuracy"].mean()
desviacion = resultados_df["accuracy"].std()

# 7. Guardar los resultados
resultados_df.to_csv(RESULTS_FILE, index=False)

# 8. Registrar el resumen en MLflow
with mlflow.start_run(run_name="train_summary_5_seeds"):

    mlflow.log_metric("accuracy_mean", float(media))
    mlflow.log_metric("accuracy_std", float(desviacion))
    mlflow.log_param("number_of_seeds", len(seeds))

# 9. Mostrar resultados
print("RESULTADOS DE LAS 5 SEMILLAS")
print(resultados_df.to_string(index=False))

print(f"\nAccuracy promedio: {media:.4f}")
print(f"Desviación estándar: {desviacion:.4f}")

print("\nArchivo results.csv actualizado.")
print("Experimentos registrados en MLflow.")

Overwriting train.py


In [11]:
# Paso 7: Ejecutar train.py para entrenar con 5 semillas y registrar los resultados en MLflow
!python train.py

RESULTADOS DE LAS 5 SEMILLAS
 seed  accuracy
    0  0.982456
   12  0.991228
   21  0.991228
   42  0.982456
  123  0.973684

Accuracy promedio: 0.9842
Desviación estándar: 0.0073

Archivo results.csv actualizado.
Experimentos registrados en MLflow.


In [12]:
# Paso 8: Registrar las versiones exactas de las librerías
from importlib.metadata import version
from pathlib import Path
import sys

librerias = ["numpy", "pandas", "scikit-learn", "mlflow"]

contenido = "\n".join(
    f"{libreria}=={version(libreria)}"
    for libreria in librerias
)

Path("requirements.txt").write_text(contenido + "\n")

print("Versión de Python:", sys.version.split()[0])
print("\nContenido de requirements.txt:")
print(Path("requirements.txt").read_text())

Versión de Python: 3.13.16

Contenido de requirements.txt:
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
mlflow==3.17.0



In [13]:
# Paso 9: Crear README.md con la documentación del experimento

from pathlib import Path
import hashlib
import sys

archivo_datos = Path("breast_cancer.csv")
huella = hashlib.sha256(archivo_datos.read_bytes()).hexdigest()
version_python = sys.version.split()[0]

readme = f"""# Assignment 5 - Reproducibility in Machine Learning

## Objetivo

Construir un experimento reproducible de clasificación con
Machine Learning, utilizando múltiples semillas, control de
versiones, prevención de data leakage y registro en MLflow.

## Dataset

- Nombre: Breast Cancer Wisconsin Diagnostic
- Fuente: Dataset disponible en scikit-learn
- Archivo: breast_cancer.csv
- Registros: 569
- Variables predictoras: 30
- Variable objetivo: target
- SHA-256: {huella}

## Modelo

- Algoritmo: LogisticRegression
- Preprocesamiento: StandardScaler
- División: 80% entrenamiento y 20% prueba
- División estratificada: Sí
- Semillas: 0, 12, 21, 42, 123
- max_iter: 1000

El preprocesamiento se realiza dentro de un Pipeline.
De esta forma, StandardScaler se ajusta solamente con
los datos de entrenamiento para evitar data leakage.

## Resultados

Se realizaron cinco ejecuciones con distintas semillas.

- Accuracy promedio: aproximadamente 0.9842 (98.42%)
- Desviación estándar: aproximadamente 0.0073

Los resultados individuales están en results.csv.

## Entorno

- Python: {version_python}
- Librerías: consultar requirements.txt
- Registro de experimentos: MLflow

## Cómo reproducir el experimento

1. Descargar los archivos de este proyecto.
2. Abrir una terminal dentro de la carpeta del proyecto.
3. Instalar las dependencias:

    pip install -r requirements.txt

4. Ejecutar el entrenamiento:

    python train.py

5. Revisar los resultados generados en results.csv.
6. Consultar los experimentos registrados en mlflow.db.

## Archivos principales

- breast_cancer.csv: dataset utilizado.
- train.py: entrenamiento con cinco semillas y MLflow.
- results.csv: resultados de las cinco ejecuciones.
- mlflow.db: historial de experimentos registrado.
- requirements.txt: versiones exactas de las librerías.
- README.md: documentación del proyecto.
- Notebook .ipynb: desarrollo paso a paso en Colab.

## Reproducibilidad

El proyecto utiliza semillas controladas, un Pipeline
para evitar data leakage, resultados estadísticos,
registro de ejecuciones y versiones de librerías.

Este experimento tiene fines académicos y no constituye
una herramienta de diagnóstico médico.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("Archivo README.md creado correctamente.")
print("Versión de Python documentada:", version_python)
print("Huella SHA-256 del dataset:", huella)

Archivo README.md creado correctamente.
Versión de Python documentada: 3.13.16
Huella SHA-256 del dataset: 75a1d74a59df9cb0a78ecc0a36085bf8a85239557f178d3b8d758e88b26416e1
